<a href="https://colab.research.google.com/github/shivanvithajayam/1BF24CS127_AI_LAB/blob/main/prepositional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:

import re
def PL_TRUE(expr, model):
    expr = expr.replace('\\', '').replace(' ', '')
    expr = expr.replace('<->', '↔').replace('->', '→')

    for s, v in model.items():
        expr = re.sub(r'\b' + s + r'\b', str(v), expr)

    def evaluate(s):
        s = s.strip()

        # Remove outer parentheses only if they enclose the whole expression
        while s.startswith('(') and s.endswith(')'):
            depth = 0
            valid = True
            for i, ch in enumerate(s):
                if ch == '(':
                    depth += 1
                elif ch == ')':
                    depth -= 1
                if depth == 0 and i != len(s) - 1:
                    valid = False
                    break
            if valid:
                s = s[1:-1]
            else:
                break

        if s.startswith('~'):
            return not evaluate(s[1:])

        for op in ['↔', '→', '|', '&']:
            depth = 0
            for i in range(len(s)):
                if s[i] == '(':
                    depth += 1
                elif s[i] == ')':
                    depth -= 1
                elif depth == 0 and s[i] == op:
                    a = evaluate(s[:i])
                    b = evaluate(s[i+1:])

                    if op == '↔':
                        return a == b
                    if op == '→':
                        return (not a) or b
                    if op == '|':
                        return a or b
                    if op == '&':
                        return a and b

        if s == 'True':
            return True
        if s == 'False':
            return False

        raise ValueError("Invalid expression: " + s)

    return evaluate(expr)




def TT_CHECK_ALL(KB, alpha, symbols, model):
    if not symbols:
        return not PL_TRUE(KB, model) or PL_TRUE(alpha, model)

    P = symbols[0]
    rest = symbols[1:]

    return (TT_CHECK_ALL(KB, alpha, rest, {**model, P: True})
            and TT_CHECK_ALL(KB, alpha, rest, {**model, P: False}))


def TT_ENTAILS(KB, alpha, symbols):
    return TT_CHECK_ALL(KB, alpha, symbols, {})


KB = input("Enter KB: ")
alpha = input("Enter Query: ")

symbols = sorted(set(re.findall(r'\b[A-Z]\b', KB + alpha)))

if TT_ENTAILS(KB, alpha, symbols):
    print("KB entails alpha")
else:
    print("KB does not entail alpha")


Enter KB: (A | C) & (B | ~C)
Enter Query: A | B
KB entails alpha
